# 01 — Обзор исследования и метод

Сравнение **способов изменения TPE** на 2D benchmark-функциях. Полное описание архитектуры —
в `docs/DESIGN.md`. Здесь — краткая суть и проверка, что код считается.

## Сравниваемые алгоритмы (каждая модификация = один фактор)

| algorithm | модификация |
|---|---|
| `random` | — (нижняя граница) |
| `tpe` | базовый TPE (rank-based l(x)/g(x), адаптивная ширина ядра + prior) |
| `tpe_gradw` | + взвешивание наблюдений по 1/‖∇f‖ (истинный градиент в реальной точке) |
| `tpe_gp` | + GP-переранжирование кандидатов |
| `tpe_gradw_gp` | + оба (это «gTPE») |
| `optuna` | внешний референс TPE |

Дополнительные факторы: `scale ∈ {raw, norm}` (нормализация), `data ∈ {clean, noisy_y}` (шум в y).

In [ ]:
# Настройка: находим корень проекта tpe_study и добавляем src в путь.
# В Colab сначала склонируйте репозиторий (раскомментируйте):
# !git clone -b claude/quirky-euler-e3laj1 https://github.com/darya-09/k5.git
# %cd k5/tpe_study
import sys, os, json
from pathlib import Path
ROOT = Path.cwd()
if not (ROOT/'src'/'tpe_study').exists():
    for c in [Path('tpe_study'), Path('k5/tpe_study'), Path('/content/k5/tpe_study')]:
        if (c/'src'/'tpe_study').exists(): ROOT=c; break
sys.path.insert(0, str((ROOT/'src').resolve()))
print('ROOT =', ROOT.resolve())

In [ ]:
# Иллюстрация: запустим baseline TPE и optuna на Sphere и сравним сходимость.
import numpy as np, matplotlib.pyplot as plt
from tpe_study.tpe import TPE
from tpe_study.baselines import optuna_tpe
from tpe_study.functions import BENCHMARKS
b = BENCHMARKS['Sphere']
obj = lambda x: (float(b.f(x)), b.grad(x))
r1 = TPE(bounds=b.bounds, n_init=10, gamma=0.15, n_candidates=24, seed=0).optimize(obj,80)
r2 = optuna_tpe(obj, b.bounds, 80, 10, 24, 0)
for r,l in [(r1,'tpe'),(r2,'optuna')]:
    plt.plot(np.minimum.accumulate(r['y_history']), label=l)
plt.yscale('log'); plt.xlabel('iter'); plt.ylabel('best f'); plt.legend(); plt.title('Sphere'); plt.show()